# A Verified 22.68 Portfolio for Verifiable Quantum Circuit Compilation

**Public score: 22.68** (geometric mean of the three task ratios), assembled from
individually verified, publicly-released component circuits.

This notebook does three things:

1. Lays out the **actual scoring model** the verifiers implement — most of the
   leverage in this competition comes from understanding where cost hides.
2. **Hash-verifies and assembles** the best verified public circuit for each task
   into a valid `submission.zip` (you can submit the output directly).
3. Documents **provenance** — every component is Apache-2.0 public work from the
   Kaggle community; the value here is careful verification and integration, and
   a map of where the remaining headroom is.

---

## 1. The scoring model (what actually matters)

The score is a **geometric mean of three improvement ratios**
(baseline cost / submission cost). A geometric mean is maximized by fixing the
*weakest* ratio — improvements multiply equally, so a 10% cut on the weakest
task is worth exactly as much as 10% on the strongest.

| Task | Cost model | What dominates | Public best |
|---|---|---|---|
| **invsqrt** | `Q × (Toffoli + rotation cost)` | lookup size k and Horner depth | ~20.2× |
| **dqi** | `repetition × Q² × T` | **qubits are quadratic**; decoder T | ~58.9× |
| **dfthcsos** | `λ_eff × T_total × Q_total` | λ_eff ~47-50, T~17k, Q~430 | ~9.65× |

Two non-obvious consequences that drove every decision:

* **`Q²` on dqi.** Shaving qubits beats shaving Toffolis 2:1 at the margin —
  the 889-qubit Schur decoder with ~434k executed Toffolis outscores decoders
  with fewer T but more qubits.
* **The repetition factor punishes any observed failure.** `verify` inflates
  cost by `rep > 1` when shots fail — we measured a candidate dqi that had
  *fewer* Toffolis but **1 value error in 102,400 shots** and it scored
  *worse* on the leaderboard. Never trade a zero-error circuit for a
  slightly-cheaper errorful one.
* **Every scratch qubit must return to |0⟩** and phases are checked exactly —
  the verifier counts measurement-induced phase flips (`HMR`/`R` on a
  non-zero qubit flips the tracked sign with probability ½).

## 2. Component portfolio — all figures are official-verifier outputs

| Task | Source / lineage | Qubits | Verified cost | Ratio |
|---|---|---|---|---|
| invsqrt | busyaprime/mrmeows merge-SAT line (63q, C=1334.5) | 63 | 102,664 | **20.2×** |
| dqi | heliosli/SomeJoint confluent Schur-family decoder | 889 | 4.148×10¹¹ | **58.9×** |
| dfthcsos | somejoint R8/B143/C62 fit + Nilanjan column gauge, J=5 loaders | 430 | 3.477×10⁸ | **9.65×** |

Geometric mean = (20.2 × 58.9 × 9.65)^⅓ ≈ **22.68** — matching the
leaderboard return on this exact file set.

Each `*.verify.json` in the attached dataset is the **unmodified JSON written
by the competition's own verifier** on this machine — shots, error bounds,
Toffoli counts, qubit counts, and the circuit's SHA-256 included. The
assembly below re-checks every file against those digests before zipping, so
what you submit is byte-identical to what was verified.

## 3. Assemble `submission.zip` (hash-verified)

The cell locates the three components anywhere under `/kaggle/input`,
verifies each file's SHA-256 against the digests recorded at verification
time, and writes a competition-valid `submission.zip` to `/kaggle/working`.

In [ ]:
import hashlib, json, shutil, zipfile
from pathlib import Path

# Digests recorded from the official verifier runs on this exact file set.
EXPECTED = {
    "dqi.kmb": "4ee7a17b4cbca4053a19bc951e7f2ef91fabdbb036006c9d658adcff4fe76da4",
    "invsqrt.kmb": "07314803a0925970cf45007ebbdb8a146378c242e3991c70d6f7509af3d583da",
    "dfthcsos/angle_k_0.kmb": "5255d0d66795e2fa1e1e1b36416237e44cf21aad2e5479c40f87e1969403eca2",
    "dfthcsos/angle_k_1.kmb": "d262661e351a90a223d626a28aaa102d8431dad11bdb383dba356e4185aabfcd",
    "dfthcsos/angle_k_10.kmb": "bd98550e656553014cb1a13cae825666c5e68a0e7a384d5965107599f128e0f0",
    "dfthcsos/angle_k_11.kmb": "dd0551f1da64eec6dc73ec7688ea7f8c6418a216b8a9125eea486a2cf939ffd2",
    "dfthcsos/angle_k_12.kmb": "c34e9459efe42c4b3679d4e2a66ffd3b6556965fd462f5f2675772cd529a26f3",
    "dfthcsos/angle_k_13.kmb": "3e9a4e1b764043e22f3587675d2b578aca04df75425b3a66615e1a1088be600f",
    "dfthcsos/angle_k_14.kmb": "8bf80f72edf85122666c68a6e8af3d0a882003836478bc66418cbc87bac1a135",
    "dfthcsos/angle_k_2.kmb": "37d30f21a92aec50287ae7085b84d2cd7a490a1c961decfafe39b9b5c6776f2e",
    "dfthcsos/angle_k_3.kmb": "f3fce956a2a213bd650687cf8d329fceab2327c19cf43bacd05dddccf73882a0",
    "dfthcsos/angle_k_4.kmb": "1f949ad54757dfaafb6bfdac99894bffd0410d924b0e454d1aec3d894dd5afbd",
    "dfthcsos/angle_k_5.kmb": "84655d0b97912178da383df083a3424c3822f23e590abc95dcaaea0e998c2e79",
    "dfthcsos/angle_k_6.kmb": "27875555d228d50e3681676b63d84d638750afb4245f23a71b67d2f6a2cff1c5",
    "dfthcsos/angle_k_7.kmb": "49e43cd23195ec70278ca2d0e618c585b8a49189d4584ee59bd0a793a720f4fb",
    "dfthcsos/angle_k_8.kmb": "0e49460864fbb5270b8755d98e3d5af14b1b94d35d183dc842d1953253cfe845",
    "dfthcsos/angle_k_9.kmb": "20b1d32646f1965669ddd5c310779d7a0b23cfdf6276fdb5a415a75359c7fc09",
    "dfthcsos/submission.npz": "7213867fe0b4bb69dd3a5c9e7961ec442bbcce2f420b12c51f8bac7f6db1ee75"
}

def sha256(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()

WORK = Path("/kaggle/working")
SUB  = WORK / "submission"
if SUB.exists():
    shutil.rmtree(SUB)

# --- locate inputs anywhere under /kaggle/input ---
by_name = {}
for p in Path("/kaggle/input").rglob("*"):
    if p.is_file():
        by_name.setdefault(p.name, p)

missing, bad = [], []
print("verifying component digests:")
for rel, want in EXPECTED.items():
    name = rel.split("/")[-1]
    src = by_name.get(name)
    if src is None:
        missing.append(rel); continue
    got = sha256(src)
    status = "ok  " if got == want else "FAIL"
    print(f"  {status} {rel:28s} {got[:16]}...")
    if got != want:
        bad.append((rel, want, got))
        continue
    dest = SUB / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(src, dest)

if missing:
    print("\nMISSING (attach the portfolio dataset):", missing)
if bad:
    print("\nHASH MISMATCHES - refusing to build:", [b[0] for b in bad])
if missing or bad:
    raise SystemExit("assembly aborted: inputs did not match verified digests")

zpath = WORK / "submission.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(SUB.rglob("*")):
        if f.is_file():
            z.write(f, f.relative_to(SUB))

names = zipfile.ZipFile(zpath).namelist()
print(f"\nwrote {zpath} ({zpath.stat().st_size/1e6:.1f} MB)")
assert "invsqrt.kmb" in names and "dqi.kmb" in names and "dfthcsos/submission.npz" in names
print("layout OK - ready to submit")

## 4. Evidence — the actual verifier records

These are the unmodified verifier outputs bundled with the dataset. Note the
dqi entry: `value_errors: 0`, `phase_errors: 0` across 102,400 shots — the
zero-error property that the repetition-factor math rewards.

In [ ]:
import json
from pathlib import Path

for jf in Path("/kaggle/input").rglob("*.verify.json"):
    rec = json.loads(jf.read_text())
    keep = {k: rec.get(k) for k in (
        "status","score","num_qubits","toffoli_count","verified_shots",
        "value_errors","phase_errors","repetition_factor",
        "lambda_eff","error_1","error_2","circuit_sha256") if k in rec}
    print(jf.name); print(json.dumps(keep, indent=2)[:1400]); print()

# implied geometric mean from the verified costs
BASE = {"invsqrt": 2_076_774.0, "dqi": 2.4425e13, "dfthcsos": 3.355599588e9}
try:
    scores = {}
    for jf in Path("/kaggle/input").rglob("*.verify.json"):
        rec = json.loads(jf.read_text())
        for task in BASE:
            if jf.name.startswith(task) and rec.get("status") == "PASS":
                scores[task] = BASE[task] / rec["score"]
    if len(scores) == 3:
        gm = (scores["invsqrt"] * scores["dqi"] * scores["dfthcsos"]) ** (1/3)
        print(f"implied GM from verifier records: {gm:.2f}x "
              f"(invsqrt {scores['invsqrt']:.2f} x dqi {scores['dqi']:.2f} x chem {scores['dfthcsos']:.2f})")
except Exception as e:
    print("gm calc skipped:", e)

## 5. What we learned from verification (the non-obvious bits)

* **Measurement-based uncomputation is not free.** `HMR` on a qubit holding
  `|1⟩` injects a coin-flip `(-1)` phase; the simulator tracks it, and
  unmatched fixups show up as `phase_errors`. Every erasure on live data must
  be reversible-uncompute or a matched `del_*` pair.
* **Re-encoded angle words are not idempotent.** Decoding and re-quantizing
  `angle_words` flips ~68% of words and can destroy a tuned fit — always carry
  quantized state forward; never round-trip through decoded vectors.
* **Cost-model cliffs are real.** On dfthcsos, `B ≥ 129` drops inner-prep
  temporary qubits 575 → 304; `brot ≥ 9` doubles angle-register cost. Shape
  (R·B) sets word count → T; C sets approximation slack.
* **`Circuit.to_file` writes text `.kmx` by default.** Binary submissions need
  `format='kmb'` — text files named `.kmb` are rejected.

## 6. Provenance & license

Components are assembled from Apache-2.0 public releases by the community —
see the attached verifier records for per-file digests. Credit: the
invsqrt merge/SAT-compiled line (busyaprime/mrmeows), the SomeJoint/heliosli
confluent DQI decoder family, and the somejoint R8/B143/C62 FeMoCo
factorization with Nilanjan's column gauge. This notebook's value-add is the
verification discipline, integration, and the scoring-model map above —
all derived from the public verifiers' own outputs.